In [106]:
import re
import httpx
import os
from dotenv import load_dotenv
from ollama import Client

_ = load_dotenv()


In [107]:
# Connect to Ollama running locally
ollama_url = 'http://localhost:11434'
client = Client(host=ollama_url)
print(client.__getstate__())


{'_client': <httpx.Client object at 0x10e598b00>}


In [108]:
# Ask the local model
model_name = 'gemma3:1b'
chat_completion = client.chat(
    model=model_name,
    messages=[{
        "role": "user",
        "content": "Hello world"
    }]
)

In [109]:
# Print the response
print(chat_completion.message.content)

Hello there! 😊 How can I help you today?


In [110]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []

        if self.system:
            self.messages.append({"role": "system", "content": self.system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat(
            model=model_name,
            messages=self.messages)
        return completion.message.content

In [111]:
# Introducing prompt
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 37 + 20
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Border Collie
returns average weight of a dog when given the breed as a number

Example session:

Question: I have a Border Collie and a Scottish Terrier. What is their combined weight?
Thought: I need to find the weight of each dog and then add them together
Action: average_dog_weight: Border Collie
PAUSE

You will be called again with this:

Observation: 37

You then output:

Thought: Now I need to find the weight of the Scottish Terrier
Action: average_dog_weight: Scottish Terrier
PAUSE

You will be called again with this:

Observation: 20

You then output:

Thought: Now I need to add 37 + 20
Action: calculate: 37 + 20
PAUSE

You will be called again with this:

Observation: 57

You then output:

Answer: A Border Collie weighs 37 lbs and a Scottish Terrier weighs 20 lbs. Their combined weight is 57 lbs.
""".strip()

In [112]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if "Scottish Terrier" in name:
        return "20"
    elif "Border Collie" in name:
        return "37"
    elif "Toy Poodle" in name:
        return "7"
    else:
        return "50"

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [113]:
abot = Agent(prompt)

In [114]:
result = abot("How much does a Toy Poodle weigh?")
print(result)

Thought: I need to determine the average weight of a Toy Poodle. I’ll consult a reliable source.
Action: average_dog_weight: Toy Poodle
PAUSE


In [115]:
result = average_dog_weight("Toy Poodle")
result

'7'

In [116]:
next_prompt = "Observation: {}".format(result)
print(f"next prompt: {abot(next_prompt)}")
print(f"messages: {abot.messages}")
abot = Agent(prompt)

next prompt: Thought: Okay, the observation is 7. I need to determine the average weight of a Toy Poodle. I'll consult a reliable source.
Action: average_dog_weight: Toy Poodle
PAUSE
messages: [{'role': 'system', 'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 37 + 20\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Border Collie\nreturns average weight of a dog when given the breed as a number\n\nExample session:\n\nQuestion: I have a Border Collie and a Scottish Terrier. What is their combined weight?\nThought: I need to find the weight of 

In [117]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I need to calculate the total weight of both dogs.\nAction: calculate: 2 + 2\nPAUSE'

In [118]:
next_prompt = "Observation: {}".format(abot(average_dog_weight("Border Collie")))
abot(next_prompt)

'Thought: I have the combined weight of the border collie and the scottish terrier, which is 37.\nAction: answer: A Border Collie weighs 37 lbs and a Scottish Terrier weighs 20 lbs. Their combined weight is 57 lbs.'

In [119]:
abot(next_prompt)

"Thought: I've calculated the combined weight of the two dogs.\nAction: answer: A Border Collie weighs 37 lbs and a Scottish Terrier weighs 20 lbs. Their combined weight is 57 lbs."

In [120]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: 20


In [121]:
abot(next_prompt)

'Thought: I need to calculate the total weight of the two dogs.\nAction: calculate: 20\nPAUSE'

In [122]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [123]:
abot(next_prompt)

'Thought: I’ve calculated the total weight of the two dogs.\nAction: answer: A Border Collie weighs 37 lbs and a Scottish Terrier weighs 20 lbs. Their combined weight is 57 lbs.'

### Add loop

In [124]:
action_re = re.compile(r'^Action: (\w+): (.*)$')

In [125]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)

        # Check if we have a final answer
        if "Answer:" in result:
            print("Final answer received!")
            return result

        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [126]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I need to find the weight of each dog and then add them together.
Action: average_dog_weight: Border Collie
PAUSE

Thought: Now I need to find the weight of the Scottish Terrier
Action: average_dog_weight: Scottish Terrier
PAUSE

Answer: A Border Collie weighs 37 lbs and a Scottish Terrier weighs 20 lbs. Their combined weight is 57 lbs.
Final answer received!


'Thought: I need to find the weight of each dog and then add them together.\nAction: average_dog_weight: Border Collie\nPAUSE\n\nThought: Now I need to find the weight of the Scottish Terrier\nAction: average_dog_weight: Scottish Terrier\nPAUSE\n\nAnswer: A Border Collie weighs 37 lbs and a Scottish Terrier weighs 20 lbs. Their combined weight is 57 lbs.'